# TRAMA Visual Factory — Free GPU Probe v0.1

This notebook performs **hardware qualification only** for the MUSEO ZERO open visual pipeline.

It does not download image models and does not incur paid compute by itself.

Target: free Colab runtime with a CUDA GPU suitable for the first quantized Qwen-Image/Edit probe.

In [ ]:
import json, os, shutil, subprocess, sys

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_present = shutil.which('nvidia-smi') is not None
gpu_name = None
vram_gib = 0.0

if gpu_present:
    gpu_name = sh("nvidia-smi --query-gpu=name --format=csv,noheader | head -1")
    vram_mib = float(sh("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -1"))
    vram_gib = vram_mib / 1024

mem_kib = int(sh("awk '/MemTotal/ {print $2}' /proc/meminfo"))
ram_gib = mem_kib / 1024 / 1024
disk = shutil.disk_usage('/content' if os.path.exists('/content') else '/')
disk_free_gib = disk.free / (1024**3)

thresholds = {
    'cuda_gpu_required': True,
    'preferred_vram_gib': 14,
    'preferred_ram_gib': 24,
    'required_free_disk_gib': 35,
}

qualified = bool(
    gpu_present and
    vram_gib >= thresholds['preferred_vram_gib'] and
    ram_gib >= thresholds['preferred_ram_gib'] and
    disk_free_gib >= thresholds['required_free_disk_gib']
)

receipt = {
    'schemaVersion': 'trama.visual-factory.free-gpu-probe/v0.1',
    'candidateId': 'MZ-VF-001',
    'adapter': 'google-colab-free',
    'gpuPresent': gpu_present,
    'gpuName': gpu_name,
    'vramGiB': round(vram_gib, 2),
    'ramGiB': round(ram_gib, 2),
    'diskFreeGiB': round(disk_free_gib, 2),
    'thresholds': thresholds,
    'decision': 'QUALIFIED' if qualified else 'NOT_QUALIFIED',
    'paidComputeAuthorized': False,
}

print(json.dumps(receipt, indent=2))

with open('trama-visual-factory-gpu-probe.json', 'w', encoding='utf-8') as f:
    json.dump(receipt, f, indent=2)

if not qualified:
    raise SystemExit('NOT_QUALIFIED: do not download the production model package on this runtime.')

print('QUALIFIED: runtime may proceed to the separate model-execution tranche.')